# Stage 5: everything together, and the test, once

**The question.** Stages 2 to 4 each asked what one kind of information adds to a video's prediction. This stage puts
together the parts that earned a place, asks on validation whether the whole beats its best part, freezes every choice,
and then opens the test split, once, for every frozen model. `lift` is stage 1's target: the video's log share rate
minus the median of its creator's last 30 videos posted at least 14 days earlier.

The parts, as the earlier stages left them (validation, seen creators):

| Part | Stage | What it is | What it did |
|---|---|---|---|
| Metadata (B3) | 2 | ridge on the video's length, wordless, the creator's earlier videos, the days since the previous one and the weekday (alpha 0.1) | the only baseline that beat zero lift: +0.0007 log10, all of it the length |
| The words | 3 and 4 | Qwen3-Embedding-4B's first 1,024 numbers for each of seven text fields, and seven "the field holds text" flags, in ridge on what B3 leaves (alpha 100) | cut B3's error by 0.0097, 2.6% |
| Recent form (B1) | 2 | an exponentially weighted mean of the creator's settled log share rates, half-life 20 videos, minus the baseline | missed the MAE bar (+0.0025 [−0.0005, +0.0053] over zero lift), and was the best ranker: Spearman 0.146 inside a creator, ROC-AUC 0.588 |
| The pixels | 4 | SigLIP 2's vector of the first frame, in ridge on what B3 and the words leave (alpha 300) | added nothing: +0.0003 [−0.0002, +0.0008] |

Recent form has never been combined with anything. Stage 2 left it as a ranking signal no baseline used well, and stage 3
found that it ranks a creator's videos inside the kept clips and inside the retired ones better than the words do
(Spearman 0.242 and 0.215, against bge-small's 0.121 and 0.186): it knows something the words don't, how the creator is
doing lately. Stage 2's hand-off said a model that fits it with the others is a new model and needs its own bar, written
before its first run. Section 0 is that bar.

The pixels stay out of the combined model, as stage 4's bar decided. Stage 4's frozen image model still goes to the test
with the others, so the test says whether "the frame adds nothing" holds on videos no choice was ever made on.

The code is `src/sharerate/` (the loader, the history features, the text cache, the frames, `RidgeGram`, the bootstrap
and the metrics), with the stacking written out in the cells.

## 0. The bar for the combined model, written before its first run (rule 7)

This cell was written, and committed, before any combined model was fitted or scored.

- **The rows.** Every model is fitted on the training cell (74,552 videos, 148 creators), every setting is chosen on the
  21,332 validation rows, and the bar is judged on the seen creators' 13,646. The test split stays closed until every
  choice below is frozen.
- **The parts are frozen as their stages left them:** B3 ridge (notebook 02: no hour, alpha 0.1); the words,
  Qwen3-Embedding-4B ridge on what B3 leaves (notebook 04_image: alpha 100, 7,175 columns, solved by `RidgeGram`); recent
  form, `history.ewm_form` with a half-life of 20 videos (notebook 02's choice), which has no value where a video has
  fewer than 10 settled earlier videos. Nothing in them is chosen again here.
- **Out of fold on the training rows.** The combiner learns how far to trust each part from the training rows, so it must
  see the predictions those rows would get from parts that never saw them. B3 and the words head are refitted in
  notebook 04_image's 5 folds grouped by creator (`GroupKFold` over the training cell, penalties fixed), and each training
  row gets the predictions of fits that never saw its creator. Recent form is fitted to no label, so it needs no folds.
  Validation and test rows get the full fits.
- **Two combiners, fixed now; validation picks one.**
  1. **Stacked, linear:** least squares, with no penalty, of lift on four columns: B3's prediction, the words head's
     correction (its prediction minus B3's), recent form (0 where it has no value) and a flag for "fewer than 10
     settled earlier videos".
  2. **One boosted model:** scikit-learn's `HistGradientBoostingRegressor`, fitted to lift on B3's five inputs (log
     duration, wordless, log earlier videos, log days since the previous video, the weekday as a number), recent form
     (missing where it has no value, so the trees learn where those rows go) and the words head's correction. The words
     go in as the head's one number: 7,175 columns are too many for small trees, and the head already turns them into
     lift. Fixed: learning rate 0.05, at most 15 leaves a tree, at least 200 videos a leaf, an L2 penalty of 1, squared
     error, no early stopping, seed 2026. Only the number of trees (25 to 1,000, every 25) is chosen, by validation MAE.

  The combined model is the one with the lower validation MAE over all 21,332 rows.
- **The primary bar.** The combined model beats the stage before, stage 4's reference (B3 + 4B ridge), if the 95% paired
  creator-bootstrap interval of its validation MAE gain over the reference lies above 0 for **seen** creators, **with
  and without the 73 reposts** (2,000 draws, seed 2026).
- **The leak check,** stages 3 and 4's: inside each creator, the combined model's correction (its prediction minus the
  reference's) regressed on whether the platform kept the clip and on the lift, seen creators. If the kept flag's
  interval lies above 0, the model finds kept clips by something other than the lift, and that is reported as a leak,
  not a pass.
- **Reported, not required:** new and held-out creators; the skill score, Spearman inside a creator, ROC-AUC, PR-AUC
  and the top-10% share. Recent form was stage 2's best ranker, so a gain in ranking without one in MAE is reported as
  exactly that, and is not a pass.
- **The ablation is descriptive.** The chosen combiner is refitted on every subset of the three parts (each alone, each
  pair, all three), and each subset's validation MAE is compared with the combined model's, seen creators, with paired
  intervals: adding the parts in turn (B3, then the words, then recent form) and taking each one out. It says what each
  part carries; it is not another chance to pass.
- **If the combined model misses,** the final model is the reference, and this notebook says recent form adds nothing
  beyond the metadata and the words. Either way the combined model goes to the test, frozen, with the others.
- **What goes to the test, and the bar for the final claim,** are written in this notebook after this section's run and
  before the test is opened.
- **A miss is a miss.** A gain whose interval touches 0 is reported as a miss, never tuned until it passes.